<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/main/STEP_6_Q_LEARNING_IMPLEMENTATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STEP 6: Q-LEARNING IMPLEMENTATION**

# Dual-Goal Reward Fuction

In [ ]:
baseline_profit = train_df["Order Profit Per Order"].mean()
baseline_days = train_df["Days for shipping (real)"].replace(0, np.nan).mean()

min_profit = baseline_profit * 1.00      # at least no profit loss
max_days = baseline_days * 0.90          # at least 10% faster
min_orders = 10

action_stats = train_df.groupby("action").agg(
    Orders=("Order Id", "count"),
    Avg_Profit=("Order Profit Per Order", "mean"),
    Avg_Days=("Days for shipping (real)", "mean")
).reset_index()

baseline_profit = train_df["Order Profit Per Order"].mean()
baseline_days = train_df["Days for shipping (real)"].mean()

good_actions = action_stats[
    (action_stats["Orders"] >= min_orders) &
    (action_stats["Avg_Profit"] >= baseline_profit * 0.98) &
    (action_stats["Avg_Days"] <= baseline_days * 0.98)
]["action"]

train_df = train_df[train_df["action"].isin(good_actions)].copy()

print("Actions retained:", train_df["action"].nunique())
print("Rows retained:", len(train_df))

In [ ]:
target_profit = train_df["Order Profit Per Order"].mean() * 1.10
target_days = train_df["Days for shipping (real)"].replace(0, np.nan).mean() * 0.80

action_check = train_df.groupby("action").agg(
    avg_profit=("Order Profit Per Order", "mean"),
    avg_days=("Days for shipping (real)", "mean"),
    count=("action", "count")
).reset_index()

print("Target profit:", target_profit)
print("Target days:", target_days)

action_check[
    (action_check["avg_profit"] >= target_profit) &
    (action_check["avg_days"] <= target_days) &
    (action_check["count"] >= 10)
].sort_values(["avg_profit", "avg_days"], ascending=[False, True])

In [ ]:
import pandas as pd
import numpy as np


# Define a small epsilon to prevent division by zero in normalization
EPSILON = 1e-8

# Aggregate metrics by action to calculate normalization values
# It's crucial to define action_summary here by grouping the current train_df
action_summary = train_df.groupby("action").agg(
    Avg_Profit=("Order Profit Per Order", "mean"),
    Avg_Days=("Days for shipping (real)", "mean"),
    Avg_Route_Mode_Cost=("Route_Mode_Cost", "mean")
).reset_index()

# Normalize Avg_Profit (higher is better)
min_profit_val = action_summary["Avg_Profit"].min()
max_profit_val = action_summary["Avg_Profit"].max()
profit_range = max_profit_val - min_profit_val
action_summary["profit_norm"] = (action_summary["Avg_Profit"] - min_profit_val) / (profit_range + EPSILON)

# Normalize Avg_Days for speed (lower days are better, so inverse scaling)
min_days_val = action_summary["Avg_Days"].min()
max_days_val = action_summary["Avg_Days"].max()
days_range = max_days_val - min_days_val
action_summary["speed_norm"] = 1 - ((action_summary["Avg_Days"] - min_days_val) / (days_range + EPSILON))

# Normalize Avg_Route_Mode_Cost for late risk (higher cost is worse, so keep direct scaling for negative coefficient)
min_cost_val = action_summary["Avg_Route_Mode_Cost"].min()
max_cost_val = action_summary["Avg_Route_Mode_Cost"].max()
cost_range = max_cost_val - min_cost_val
action_summary["late_risk_norm"] = (action_summary["Avg_Route_Mode_Cost"] - min_cost_val) / (cost_range + EPSILON)

# Calculate the action_score (reward) for each action using the specified weights
action_summary["action_score"] = (
    3.0 * action_summary["profit_norm"]
    + 2.0 * action_summary["speed_norm"]
    - 1.0 * action_summary["late_risk_norm"]
)

# Before merging, check if 'action_score' already exists in train_df and drop it
# This prevents the MergeError if 'action_score' was inadvertently present
if 'action_score' in train_df.columns:
    train_df = train_df.drop(columns=['action_score'])

# Merge the action_score (which is the reward for each action) back into train_df
# This will add the 'action_score' column to train_df
train_df = train_df.merge(
    action_summary[["action", "action_score"]],
    on="action",
    how="left"
)

# Assign the calculated action_score from the merged column to train_df["reward"]
train_df["reward"] = train_df["action_score"]


# Initialize Q-Table and Train

In [ ]:
from sklearn.preprocessing import LabelEncoder
import joblib

# The train_df is already filtered by `good_actions` from the preceding cell.
# Re-create the 'state' string for the filtered train_df
state_cols = [
    "Product Category Id_bin",
    "stocking_score_bin",
    "margin_pct_bin",
    "shipping_delay_bin",
    "Order Region_bin",
    "Shipping Cost_bin",
    "Order Item Total_bin"
]
train_df["state"] = train_df[state_cols].astype(str).agg("|".join, axis=1)

# Re-fit LabelEncoders on the filtered train_df to ensure contiguous IDs from 0
state_encoder_q_learning = LabelEncoder()
train_df["state_id"] = state_encoder_q_learning.fit_transform(train_df["state"])

action_encoder_q_learning = LabelEncoder()
train_df["action_id"] = action_encoder_q_learning.fit_transform(train_df["action"])

# Now recalculate n_states and n_actions based on the re-encoded IDs
n_states = train_df["state_id"].nunique()
n_actions = train_df["action_id"].nunique()

Q = np.zeros((n_states, n_actions))

alpha = 0.1
gamma = 0.3
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
episodes = 100

# Rebuild valid_state_actions and reward_table using the new state_id and action_id
valid_state_actions = (
    train_df.groupby("state_id")["action_id"]
    .apply(lambda x: sorted(x.unique()))
    .to_dict()
)

reward_table = (
    train_df.groupby(["state_id", "action_id"])["reward"]
    .mean()
    .to_dict()
)

state_ids = train_df["state_id"].to_numpy()

for episode in range(episodes):
    total_reward = 0

    for i in range(len(state_ids) - 1):
        state = int(state_ids[i])
        next_state = int(state_ids[i + 1])

        valid_actions_for_current_state = valid_state_actions.get(state, [])

        # Handle cases where a state might not have any valid actions (though unlikely with filtered data)
        if len(valid_actions_for_current_state) == 0:
            continue

        if np.random.rand() < epsilon:
            action = np.random.choice(valid_actions_for_current_state)
        else:
            # Ensure action is chosen from valid actions
            action = max(valid_actions_for_current_state, key=lambda a: Q[state, a])

        reward = reward_table.get((state, action), 0)

        next_valid_actions_for_next_state = valid_state_actions.get(next_state, [])

        if len(next_valid_actions_for_next_state) > 0:
            next_best_q = max(Q[next_state, a] for a in next_valid_actions_for_next_state)
        else:
            next_best_q = 0

        Q[state, action] += alpha * (
            reward + gamma * next_best_q - Q[state, action]
        )

        total_reward += reward

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % 10 == 0:
        print(f"Episode {episode}: Reward = {total_reward:.2f}")

print("Training complete")

# Save the newly fitted encoders if they are needed later for evaluation
joblib.dump(state_encoder_q_learning, "state_encoder_q_learning.pkl")
joblib.dump(action_encoder_q_learning, "action_encoder_q_learning.pkl")

# Extract the Learned Policy

In [ ]:
policy_action_ids = []

for s in range(n_states):
    valid_actions = valid_state_actions.get(s, [])

    if len(valid_actions) == 0:
        policy_action_ids.append(np.argmax(Q[s]))
    else:
        best_action = max(valid_actions, key=lambda a: Q[s, a])
        policy_action_ids.append(best_action)

policy_df = pd.DataFrame({
    "state_id": np.arange(n_states),
    "action_id": policy_action_ids,
    "recommended_action": action_encoder_q_learning.inverse_transform(policy_action_ids)
})

policy_df[
    [
        "Recommended_Fulfillment_Region",
        "Recommended_Route",
        "Recommended_Shipping_Mode"
    ]
] = policy_df["recommended_action"].str.split(" \\| ", expand=True)

policy_df.head()

# Evaluate Recommended Shipping Modes

In [ ]:
eval_df = train_df.merge(policy_df, on="state_id", how="left")

# If action contains route/mode like:
# Fulfillment_Region | Route | Shipping_Mode
eval_df["Recommended_Shipping_Mode"] = (
    eval_df["recommended_action"]
    .astype(str)
    .str.split("|")
    .str[-1]
    .str.strip()
)

# Compare historical vs recommended shipping mode
mode_comparison = pd.crosstab(
    eval_df["Shipping Mode_str"],
    eval_df["Recommended_Shipping_Mode"],
    normalize="index"
) * 100

print("Historical vs Recommended Shipping Mode (%):")
display(mode_comparison.round(2))

# Recommended mode distribution
recommended_distribution = (
    eval_df["Recommended_Shipping_Mode"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Recommended Shipping Mode Distribution (%):")
display(recommended_distribution)

# Historical mode distribution
historical_distribution = (
    eval_df["Shipping Mode_str"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Historical Shipping Mode Distribution (%):")
display(historical_distribution)

In [ ]:
recommended_mode_summary = (
    eval_df.groupby("Recommended_Shipping_Mode")
    .agg(
        orders=("Order Id", "count"),
        avg_profit=("Order Profit Per Order", "mean"),
        avg_late_risk=("Late_delivery_risk", "mean"),
        avg_shipping_days=("Days for shipping (real)", "mean"),
        avg_route_mode_cost=("Route_Mode_Cost", "mean"),
        avg_reward=("reward", "mean")
    )
    .sort_values("avg_reward", ascending=False)
)

display(recommended_mode_summary.round(4))

In [ ]:
recommended_mode_summary = (
    eval_df.groupby("Recommended_Shipping_Mode")
    .agg(
        orders=("Order Id", "count"),
        avg_profit=("Order Profit Per Order", "mean"),
        avg_late_risk=("Late_delivery_risk", "mean"),
        avg_shipping_days=("Days for shipping (real)", "mean"),
        avg_route_mode_cost=("Route_Mode_Cost", "mean"),
        avg_reward=("reward", "mean")
    )
    .sort_values("avg_reward", ascending=False)
)

display(recommended_mode_summary.round(4))

# Measure Improvement

In [ ]:
# Compare historical Q-value vs recommended Q-value

eval_df["historical_q"] = [
    Q[s, a] for s, a in zip(eval_df["state_id"], eval_df["action_id_x"])
]

eval_df["policy_q"] = [
    Q[s].max() for s in eval_df["state_id"]
]

q_results = pd.DataFrame({
    "Metric": ["Avg Q-Value"],
    "Historical": [eval_df["historical_q"].mean()],
    "Recommended Policy": [eval_df["policy_q"].mean()]
})

q_results["Change"] = (
    q_results["Recommended Policy"] - q_results["Historical"]
)

q_results["Percent Change"] = (
    q_results["Change"] / q_results["Historical"].replace(0, np.nan)
) * 100

display(q_results.round(4))

In [ ]:
comparison = pd.crosstab(
    eval_df["Shipping Mode_str"],
    eval_df["recommended_action"].str.split("|").str[-1].str.strip(),
    normalize="index"
) * 100

display(comparison.round(2))

In [ ]:
import joblib
import pandas as pd
import numpy as np

# Load the specific encoders used for the Q-learning process
state_encoder_q_learning = joblib.load("state_encoder_q_learning.pkl")
action_encoder_q_learning = joblib.load("action_encoder_q_learning.pkl")

# The eval_df comes from merging with the filtered train_df in the previous cell.
# However, to explicitly ensure state_ids in eval_df are consistent with the
# Q-table dimensions, we re-encode them using state_encoder_q_learning.

# Define state_cols used for creating the state string
state_cols = [
    "Product Category Id_bin",
    "stocking_score_bin",
    "margin_pct_bin",
    "shipping_delay_bin",
    "Order Region_bin",
    "Shipping Cost_bin",
    "Order Item Total_bin"
]

# Create the 'state' string for eval_df
eval_df["state_temp"] = eval_df[state_cols].astype(str).agg("|".join, axis=1)

# Filter eval_df to only include states seen by the Q-learning encoder
# This is crucial as eval_df might contain states not present in the filtered train_df
# that state_encoder_q_learning was fitted on.
valid_q_learning_states = set(state_encoder_q_learning.classes_)
eval_df = eval_df[eval_df["state_temp"].isin(valid_q_learning_states)].copy()

# Now, transform the state_temp to state_id using the Q-learning specific encoder
eval_df["state_id"] = state_encoder_q_learning.transform(eval_df["state_temp"])

# Drop the temporary state column
eval_df = eval_df.drop(columns=["state_temp"])


# Map each state to its learned recommended action using the existing correct 'state_id'
eval_df["recommended_action_id"] = eval_df["state_id"].map(lambda s: np.argmax(Q[s]))

# Convert recommended_action_id back to action string using the correct action encoder
# This line was already fixed in the previous iteration to use action_encoder_q_learning
eval_df["recommended_action"] = action_encoder_q_learning.inverse_transform(eval_df["recommended_action_id"])

# Estimate outcomes by historical averages for each recommended action
# action_outcomes is correctly based on the filtered train_df
# Make sure train_df is the one used for Q-learning (filtered and re-encoded)
# It's currently in the kernel state, let's ensure it's referenced correctly.
action_outcomes = train_df.groupby("action")[[
    "Order Profit Per Order",
    "Days for shipping (real)"
]].mean()

eval_df = eval_df.merge(
    action_outcomes,
    left_on="recommended_action",
    right_index=True,
    suffixes=("_historical", "_policy")
)

historical_profit = eval_df["Order Profit Per Order_historical"].mean()
policy_profit = eval_df["Order Profit Per Order_policy"].mean()

historical_days = eval_df["Days for shipping (real)_historical"].mean()
policy_days = eval_df["Days for shipping (real)_policy"].mean()

profit_change_pct = ((policy_profit - historical_profit) / historical_profit) * 100
days_change_pct = ((historical_days - policy_days) / historical_days) * 100

print(f"Historical Profit: {historical_profit:.4f}")
print(f"Policy Profit: {policy_profit:.4f}")
print(f"Profit Change %: {profit_change_pct:.2f}%")

print(f"Historical Days: {historical_days:.4f}")
print(f"Policy Days: {policy_days:.4f}")
print(f"Delivery Time Reduction %: {days_change_pct:.2f}%")

if profit_change_pct >= 10 and days_change_pct >= 20:
    print("Goal achieved")
else:
    print("Goal not achieved")

In [ ]:
profit_goal = profit_change_pct > 0
delivery_goal = days_change_pct > 0

if profit_goal and delivery_goal:
    print("Goal achieved")
else:
    print("Goal not achieved")

In [ ]:
summary = train_df.groupby("action").agg(
    Profit=("Order Profit Per Order","mean"),
    Days=("Days for shipping (real)","mean"),
    Orders=("action","size")
)

summary["ProfitGain%"] = (
    (summary["Profit"] - summary["Profit"].mean())
    / summary["Profit"].mean()
) * 100

summary["DayReduction%"] = (
    (summary["Days"].mean() - summary["Days"])
    / summary["Days"].mean()
) * 100

summary.sort_values(
    ["ProfitGain%","DayReduction%"],
    ascending=False
).head(30)

In [ ]:
recommended_summary = (
    eval_df.groupby("recommended_action")
    .agg(
        Orders=("recommended_action", "size"),
        Profit=("Order Profit Per Order_policy", "mean"),
        Days=("Days for shipping (real)_policy", "mean")
    )
    .sort_values("Orders", ascending=False)
)

print(recommended_summary.head(20))

# Save Results

In [ ]:
policy_df.to_csv(
    "q_learning_policy.csv",
    index=False, encoding="latin1"
)

np.save(
    "q_table.npy",
    Q
)

print("Policy and Q-table saved")